# Config 2: QLoRA fine-tune (GPU)

Trains a QLoRA adapter on top of `google/gemma-3-1b-it` -- the same base
model config1 serves via Ollama -- so that "config1 vs config2" in the
final results table isolates the effect of fine-tuning rather than
comparing two different models. This notebook is a runnable copy of
`SETUP.md` in this folder (section numbers match); the plan-level framing
(why config2 exists, what it's being compared against) is in the repo
root's `PLAN.md`.

**Hard constraint:** training data must never overlap `../eval-programs/`
(the 20 held-out programs used for scoring). `generate_training_corpus.py`
enforces this automatically below.

Run top to bottom: get a GPU -> install deps -> generate training data ->
fine-tune (incl. Hugging Face auth) -> generate predictions -> score
base vs. tuned -> *(optional)* ODA credentials for real-SAS grounding.

## 1. Get a free GPU

Colab UI action, not a cell to run: **Runtime -> Change runtime type ->
Hardware accelerator: T4 GPU -> Save**, before running anything below.
(Kaggle Notebooks and Lightning AI Studios work the same way if Colab's
free tier is unavailable -- see `SETUP.md` section 1 for details.)

## 1b. Pull the code from GitHub into this Colab session

This notebook expects to run with its **cwd inside `config2-qlora-gpu/`**,
with `../eval-programs/` and `../results/` present as siblings (that's
what `../data`, `../adapters`, `../eval-programs`, `../results` below
resolve to). `degit` (via `npx`, already available in Colab) copies just
those folders straight from GitHub -- no full clone, no `.git` history,
and no auth needed since this repo is public.

Only pulling `eval-programs/` and `results/` because later cells
(leakage check, inference, scoring) read from them; skip that second
line if you only plan to run steps 2-4 (train + save the adapter) in
this session.

In [ ]:
!npx --yes degit patrickjlong1/sas-llm-paper/config2-qlora-gpu config2-qlora-gpu
!npx --yes degit patrickjlong1/sas-llm-paper/eval-programs eval-programs
!npx --yes degit patrickjlong1/sas-llm-paper/results results
%cd config2-qlora-gpu

## 2. Install dependencies

`trl>=0.24` is a hard floor, not just a version bump -- `qlora_finetune.py`
relies on the current `trl` API (`SFTConfig` + `assistant_only_loss` over a
`"messages"`-shaped dataset). See `requirements.txt` for why older `trl`
won't work at all on this Python.

In [1]:
!pip -q install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 87.3 MB/s eta 0:00:00


## 3. Generate training data

Synthetic corpus, same generator `eval-programs/` itself uses but a
disjoint id range/seed (`corpus_gen.py`). This step also runs
`check_no_leakage.py` automatically and would refuse to proceed on any
overlap with `../eval-programs/gold/` -- the "no leakage found" line
below is that check passing, not just a status message.

In [3]:
!python3 generate_training_corpus.py --n 600 --out-dir ../data

wrote 600 .sas programs to ../data/train_sas
wrote 600 gold JSON files to ../data/train_gold
wrote combined jsonl to ../data/train.jsonl
no leakage found: 0 eval programs, training set clean.

training corpus ready at ../data/train.jsonl -- verified clean against eval-programs/gold


## 4. Fine-tune

`google/gemma-3-1b-it` is gated on Hugging Face -- accept the license on
the model's page while logged in, then run the login cell below (or set
the `HF_TOKEN` Colab secret) before training can download the weights.
This is the same base model config1 serves via Ollama, kept matched on
purpose so "config1 vs config2" measures fine-tuning's effect, not a
different-model comparison.

QLoRA (rank 32, attention + MLP target modules) on 600 synthetic examples,
2 epochs, `--maxlen 2048`, loss on the assistant turn only. Saves the
adapter to `../adapters/sasdoc-lora` automatically once training finishes
(`trainer.model.save_pretrained` + tokenizer + a `run_config.json` of the
CLI args) -- no separate save step needed. `save_strategy="epoch"` also
means the `Trainer` drops its own checkpoint into `../adapters/sasdoc-lora/checkpoint-<step>/`
at each epoch boundary.

**Runtime:** on a free T4, ~50s/optimizer-step observed, 150 steps total
(600 examples / effective batch 8 / 2 epochs) -- so budget **1.5-2.5
hours**, not the 30-90 min some earlier estimates assumed. The eval pass
+ checkpoint write at each epoch boundary (steps 75 and 150) adds a bit
more on top.

The `WARNING: this trl's SFTConfig does not accept: warmup_ratio --
using its defaults for them` line below is expected and harmless:
`requirements.txt` only pins a floor (`trl>=0.24`), `SFTConfig`'s exact
field set drifts across `trl` releases, and `qlora_finetune.py` filters
its kwargs against whatever signature the resolved `trl` actually has
instead of crashing on a dropped one.

In [5]:
from huggingface_hub import login
login()

In [ ]:
!python3 qlora_finetune.py --train ../data/train.jsonl --model google/gemma-3-1b-it \
    --maxlen 2048 --epochs 2 --out ../adapters/sasdoc-lora

train: kept 600 / 600 at maxlen=2048
Loading weights: 100% 340/340 [00:00<00:00, 420.51it/s]
trainable params: 26,091,520 || all params: 1,025,977,472 || trainable%: 2.5431
Tokenizing train dataset: 100% 600/600 [00:02<00:00, 236.25 examples/s]
Building labels for train dataset: 100% 600/600 [00:02<00:00, 290.62 examples/s]
Truncating train dataset: 100% 600/600 [00:02<00:00, 242.26 examples/s]
Dropping fully masked examples from train dataset: 100% 600/600 [00:01<00:00, 583.20 examples/s]
Tokenizing eval dataset: 100% 20/20 [00:00<00:00, 218.08 examples/s]
Building labels for eval dataset: 100% 20/20 [00:00<00:00, 298.11 examples/s]
Truncating eval dataset: 100% 20/20 [00:00<00:00, 393.46 examples/s]
Dropping fully masked examples from eval dataset: 100% 20/20 [00:00<00:00, 590.23 examples/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated wit

### Verify & persist the adapter

Still part of step 4: confirm the save worked, then get the adapter off
Colab's disk before the runtime recycles (local storage does not survive
that). Everything below this point in the notebook is **prepped but not
yet run** -- it picks up once the fine-tune cell above finishes.

In [ ]:
!ls -la ../adapters/sasdoc-lora

In [ ]:
# Colab's local disk does NOT survive a runtime recycle -- grab the adapter
# now (it's tens of MB, not the base model). Skip this if --out already
# pointed at mounted Drive.
from google.colab import files
import shutil
shutil.make_archive("sasdoc-lora", "zip", "../adapters/sasdoc-lora")
files.download("sasdoc-lora.zip")

## 5. Generate predictions for scoring

Run **twice** on the identical 20 held-out `eval-programs/` -- once with
the adapter, once without -- so base-vs-tuned isn't confounded with a
different input set. Greedy decoding, so this is a fair comparison rather
than a sampling artifact. Each run also records per-program wall-clock
generation time in its `.meta.json` sidecars (distinct from the one-time
training cost above).

In [ ]:
!python3 infer.py --out ../results/preds/config2-base

In [ ]:
!python3 infer.py --adapter ../adapters/sasdoc-lora --out ../results/preds/config2-tuned

## 5b. Score the predictions

`results/run_eval.py` wraps `score.py` (schema validity, variable/macro/IO
precision-recall-F1, hallucination rate -- see its docstring) into the
`.scores.jsonl` files the plan's results table reads. Add `--run-judge` to
either call below if you also want the free-text description-quality
metric (needs a judge model reachable from this session).

In [ ]:
!python3 ../results/run_eval.py --config config2-base --pred-dir ../results/preds/config2-base \
    --out-prefix ../results/outputs/config2-base

In [ ]:
!python3 ../results/run_eval.py --config config2-tuned --pred-dir ../results/preds/config2-tuned \
    --out-prefix ../results/outputs/config2-tuned

### Combine into the results table

Prints the plan's actual results-table row (bootstrapped mean + CI per
metric) for each of base and tuned -- run again later with both this
config's rows and config1/config3's once those exist, to get the full
cross-config comparison.

In [ ]:
!python3 ../results/run_eval.py --table \
    --scores ../results/outputs/config2-base.scores.jsonl --meta-dir ../results/preds/config2-base \
    --label "Config 2: base (no adapter)"
!python3 ../results/run_eval.py --table \
    --scores ../results/outputs/config2-tuned.scores.jsonl --meta-dir ../results/preds/config2-tuned \
    --label "Config 2: QLoRA-tuned"

## 6. (Optional) SAS OnDemand for Academics (ODA) credentials

Not needed for the base-vs-tuned comparison above -- only for
`oda_harvest.py`, which scores against what SAS actually materializes
(via `dictionary.columns`) rather than just the generator's own spec, by
producing a `--extra-source` file for `results/score.py`. Same
account/credential setup as config1 and config3. Skip this section unless
you want that extra grounding.

Get a free ODA account first if you don't have one (SAS OnDemand for
Academics signup -- no cost, academic/non-commercial use). **Never paste
your password into a notebook cell that gets saved/shared** -- use Colab
Secrets and read them into `os.environ`, as below.

In [ ]:
# Read from Colab Secrets (the key icon in the left sidebar), not literals:
# from google.colab import userdata
# os.environ["ODA_USER"] = userdata.get("ODA_USER")
# os.environ["ODA_PASS"] = userdata.get("ODA_PASS")
import os

with open(os.path.expanduser("~/.authinfo"), "w") as f:
    f.write("oda user %s password %s\n" % (os.environ["ODA_USER"], os.environ["ODA_PASS"]))
os.chmod(os.path.expanduser("~/.authinfo"), 0o600)

In [ ]:
!apt-get -qq install default-jdk > /dev/null
!pip -q install saspy

In [ ]:
# sascfg_personal.py's iomhost list is already filled in for US-region/usw2;
# if your account is a different region, see config1's SETUP.md for the
# other two regions' host names.
import saspy, shutil, os
shutil.copy("sascfg_personal.py", os.path.dirname(saspy.__file__))

In [ ]:
!python3 oda_harvest.py --sas-dir ../eval-programs/programs --out ../data/oda_metadata.json